In [1]:
# %%
"""
Brightkite radius-of-gyration benchmark.

This script is organized as editor cells (``# %%``) so you can open it
in VS Code or Jupyter as a multi-cell workflow. It expects the Brightkite
CSV to be at `tests/shared/data/brightkite.csv`.

Cell 1: load dataset into Polars (full ~4M rows)
Cell 2: run the `skmob2` benchmark
Cell 3: run the original `skmob` benchmark (pandas conversion)
"""

# import skmob2
import time
from pathlib import Path
import polars as pl
import pandas as pd
from skmob import TrajDataFrame
import skmob
import time
import tracemalloc

In [2]:
from skmob import TrajDataFrame
from skmob.measures.individual import jump_lengths

In [4]:
DATA_PATH = "../shared/data/loc-brightkite_totalCheckins.txt.gz"

df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    header=0,
    # nrows=100_000,
    names=["uidabc", "datetime", "lat", "lng", "location id"],
)
df["datetime"] = pd.to_datetime(df["datetime"])
print(f"Loaded dataframe with {len(df)} rows and columns: {list(df.columns)[:10]}")

Loaded dataframe with 4747286 rows and columns: ['uidabc', 'datetime', 'lat', 'lng', 'location id']


In [5]:
tdf = TrajDataFrame(df, timestamp=True) # already has datetime column, so skip conversion

In [6]:
jump_lengths(tdf)

,jump_lengths
0,"[34.12862441305054, 0.3017130530943952, 0.0, 0..."


In [6]:
DATA_PATH = "../shared/data/loc-brightkite_totalCheckins.txt.gz"

df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    header=0,
    # nrows=100_000,
    names=["uidabc", "datetime", "lat", "lng", "location id"],
)
df["datetime"] = pd.to_datetime(df["datetime"])
print(f"Loaded dataframe with {len(df)} rows and columns: {list(df.columns)[:10]}")

Loaded dataframe with 4747286 rows and columns: ['uidabc', 'datetime', 'lat', 'lng', 'location id']


In [16]:
skmob2.jump_lengths(df)

,uid,jump_lengths
0,0,"[19.640494457447723, 0.0, 0.0, 1.7434335091684..."
1,1,"[6.505339409923645, 46.75443058363974, 53.9285..."
2,2,"[0.0, 0.0, 0.0, 0.0, 3.641014748771287, 0.0, 5..."
3,3,"[3861.275963494035, 4.061636923656222, 5.91633..."
4,4,"[15511.949011944975, 0.0, 15511.949011944975, ..."
...,...,...
51401,58222,[]
51402,58224,[]
51403,58225,[]
51404,58226,[]


In [8]:
tdf = TrajDataFrame(df, timestamp=True) # already has datetime column, so skip conversion

In [9]:
jump_lengths(tdf)

,jump_lengths
0,"[34.12862441305054, 0.3017130530943952, 0.0, 0..."


In [11]:
tdf = TrajDataFrame(df) # already has datetime column, so skip conversion

In [12]:
jump_lengths(tdf)

,jump_lengths
0,"[34.12862441305054, 0.3017130530943952, 0.0, 0..."


In [ ]:

from skmob2.measures.spatial.radius_of_gyration import radius_of_gyration as rog_skmob2

print("\nWarming up skmob2...")
_ = rog_skmob2(df)

times = []
peak_memories = []

for i in range(5):
    time.sleep(0.5)
    
    # Start tracking memory allocations
    tracemalloc.start()
    
    start = time.perf_counter()
    _ = rog_skmob2(df)
    end = time.perf_counter()
    
    # Capture the peak memory used during the function call
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop() # Reset tracker for the next round
    
    duration = end - start
    peak_mem_mb = peak_mem / (1024 * 1024) # Convert bytes to Megabytes
    
    times.append(duration)
    peak_memories.append(peak_mem_mb)
    
    print(f"skmob2 Round {i+1}: {duration:.4f} seconds | Peak Overhead: {peak_mem_mb:.2f} MB")

print(f"skmob2 Average Time: {sum(times)/len(times):.4f} s")
print(f"skmob2 Minimum Time: {min(times):.4f} s")
print(f"skmob2 Average Peak Overhead: {sum(peak_memories)/len(peak_memories):.2f} MB\n")


Warming up skmob2...
skmob2 Round 1: 0.4246 seconds | Peak Overhead: 6.30 MB
skmob2 Round 2: 0.4254 seconds | Peak Overhead: 6.30 MB
skmob2 Round 3: 0.4240 seconds | Peak Overhead: 6.30 MB
skmob2 Round 4: 0.4244 seconds | Peak Overhead: 6.30 MB
skmob2 Round 5: 0.5302 seconds | Peak Overhead: 6.30 MB
skmob2 Average Time: 0.4457 s
skmob2 Minimum Time: 0.4240 s
skmob2 Average Peak Overhead: 6.30 MB



In [4]:
try:
    from skmob.measures.individual import radius_of_gyration as rog_skmob
    from skmob import TrajDataFrame
except Exception as exc: 
    raise SystemExit("Unable to import scikit-mobility. Install it via pip.")

# Prepare the data exactly as scikit-mobility expects
df_skmob = df.rename(columns={"latitude": "lat", "longitude": "lng", "check-in_time": "datetime"})

print("Warming up scikit-mobility...")
# Include TrajDataFrame creation in warmup
traj_df_warmup = TrajDataFrame(df_skmob, user_id="user", latitude="lat", longitude="lng")
_ = rog_skmob(traj_df_warmup)

times_skmob = []
peak_memories_skmob = []

for i in range(5):
    time.sleep(0.5)
    
    tracemalloc.start()
    start = time.perf_counter()
    
    # Include required initialization in the timer, just like the real world
    traj_df = TrajDataFrame(df_skmob, user_id="user", latitude="lat", longitude="lng")
    _ = rog_skmob(traj_df)
    
    end = time.perf_counter()
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    duration = end - start
    peak_mem_mb = peak_mem / (1024 * 1024)
    
    times_skmob.append(duration)
    peak_memories_skmob.append(peak_mem_mb)
    
    print(f"scikit-mobility Round {i+1}: {duration:.4f} seconds | Peak Overhead: {peak_mem_mb:.2f} MB")

print(f"scikit-mobility Average Time: {sum(times_skmob)/len(times_skmob):.4f} s")
print(f"scikit-mobility Minimum Time: {min(times_skmob):.4f} s")
print(f"scikit-mobility Average Peak Overhead: {sum(peak_memories_skmob)/len(peak_memories_skmob):.2f} MB")

Warming up scikit-mobility...


100%|██████████| 51406/51406 [01:44<00:00, 493.67it/s]


scikit-mobility Round 1: 105.5391 seconds | Peak Overhead: 510.45 MB


100%|██████████| 51406/51406 [01:50<00:00, 465.10it/s]


scikit-mobility Round 2: 111.9750 seconds | Peak Overhead: 510.42 MB


...

scikit-mobility Round 3: 108.5175 seconds | Peak Overhead: 510.41 MB


100%|██████████| 51406/51406 [01:47<00:00, 480.24it/s]


scikit-mobility Round 4: 108.4854 seconds | Peak Overhead: 510.42 MB


100%|██████████| 51406/51406 [01:53<00:00, 451.27it/s]

scikit-mobility Round 5: 115.3405 seconds | Peak Overhead: 510.42 MB
scikit-mobility Average Time: 109.9715 s
scikit-mobility Minimum Time: 105.5391 s
scikit-mobility Average Peak Overhead: 510.42 MB


In [5]:
try:
    from skmob.measures.individual import radius_of_gyration as rog_skmob
    from skmob import TrajDataFrame
except Exception as exc: 
    raise SystemExit("Unable to import scikit-mobility. Install it via pip.")

# Prepare the data exactly as scikit-mobility expects
df_skmob = df.rename(columns={"latitude": "lat", "longitude": "lng", "check-in_time": "datetime"})

print("Warming up scikit-mobility...")
# Include TrajDataFrame creation in warmup
traj_df_warmup = TrajDataFrame(df_skmob, user_id="user", latitude="lat", longitude="lng")
_ = rog_skmob(traj_df_warmup)

times_skmob = []
peak_memories_skmob = []

for i in range(5):
    time.sleep(0.5)
    
    tracemalloc.start()
    start = time.perf_counter()
    
    # Include required initialization in the timer, just like the real world
    traj_df = TrajDataFrame(df_skmob, user_id="user", latitude="lat", longitude="lng")
    _ = rog_skmob(traj_df)
    
    end = time.perf_counter()
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    duration = end - start
    peak_mem_mb = peak_mem / (1024 * 1024)
    
    times_skmob.append(duration)
    peak_memories_skmob.append(peak_mem_mb)
    
    print(f"scikit-mobility Round {i+1}: {duration:.4f} seconds | Peak Overhead: {peak_mem_mb:.2f} MB")

print(f"scikit-mobility Average Time: {sum(times_skmob)/len(times_skmob):.4f} s")
print(f"scikit-mobility Minimum Time: {min(times_skmob):.4f} s")
print(f"scikit-mobility Average Peak Overhead: {sum(peak_memories_skmob)/len(peak_memories_skmob):.2f} MB")

Warming up scikit-mobility...


100%|██████████| 51406/51406 [01:43<00:00, 498.46it/s]


scikit-mobility Round 1: 104.5495 seconds | Peak Overhead: 510.42 MB


100%|██████████| 51406/51406 [01:42<00:00, 499.61it/s]


scikit-mobility Round 2: 104.3836 seconds | Peak Overhead: 510.41 MB


...

scikit-mobility Round 3: 112.1992 seconds | Peak Overhead: 510.41 MB


100%|██████████| 51406/51406 [01:43<00:00, 496.74it/s]


scikit-mobility Round 4: 104.9081 seconds | Peak Overhead: 510.41 MB


100%|██████████| 51406/51406 [01:44<00:00, 491.95it/s]

scikit-mobility Round 5: 105.9449 seconds | Peak Overhead: 510.42 MB
scikit-mobility Average Time: 106.3971 s
scikit-mobility Minimum Time: 104.3836 s
scikit-mobility Average Peak Overhead: 510.41 MB


In [ ]:
import time
import tracemalloc
from skmob2.measures.spatial.radius_of_gyration import radius_of_gyration as rog_skmob2

print("\nWarming up skmob2...")
_ = skmob2.jump_lengths(df)

times = []
peak_memories = []

for i in range(5):
    time.sleep(0.5)
    
    # Start tracking memory allocations
    tracemalloc.start()
    
    start = time.perf_counter()
    _ = skmob2.jump_lengths(df)
    end = time.perf_counter()
    
    # Capture the peak memory used during the function call
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop() # Reset tracker for the next round
    
    duration = end - start
    peak_mem_mb = peak_mem / (1024 * 1024) # Convert bytes to Megabytes
    
    times.append(duration)
    peak_memories.append(peak_mem_mb)
    
    print(f"skmob2 Round {i+1}: {duration:.4f} seconds | Peak Overhead: {peak_mem_mb:.2f} MB")

print(f"skmob2 Average Time: {sum(times)/len(times):.4f} s")
print(f"skmob2 Minimum Time: {min(times):.4f} s")
print(f"skmob2 Average Peak Overhead: {sum(peak_memories)/len(peak_memories):.2f} MB\n")


Warming up skmob2...
skmob2 Round 1: 39.8366 seconds | Peak Overhead: 573.60 MB
skmob2 Round 2: 40.8484 seconds | Peak Overhead: 573.59 MB
skmob2 Round 3: 38.0556 seconds | Peak Overhead: 573.60 MB
skmob2 Round 4: 39.8411 seconds | Peak Overhead: 573.59 MB
skmob2 Round 5: 39.6735 seconds | Peak Overhead: 573.59 MB
skmob2 Average Time: 39.6510 s
skmob2 Minimum Time: 38.0556 s
skmob2 Average Peak Overhead: 573.59 MB



In [6]:
try:
    from skmob.measures.individual import jump_lengths as rog_skmob
    from skmob import TrajDataFrame
except Exception as exc: 
    raise SystemExit("Unable to import scikit-mobility. Install it via pip.")

# Prepare the data exactly as scikit-mobility expects
df_skmob = df.rename(columns={"latitude": "lat", "longitude": "lng", "check-in_time": "datetime"})

print("Warming up scikit-mobility...")
# Include TrajDataFrame creation in warmup
traj_df_warmup = TrajDataFrame(df_skmob, user_id="user", latitude="lat", longitude="lng")
_ = rog_skmob(traj_df_warmup)

times_skmob = []
peak_memories_skmob = []

for i in range(5):
    time.sleep(0.5)
    
    tracemalloc.start()
    start = time.perf_counter()
    
    # Include required initialization in the timer, just like the real world
    traj_df = TrajDataFrame(df_skmob, user_id="user", latitude="lat", longitude="lng")
    _ = rog_skmob(traj_df)
    
    end = time.perf_counter()
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    duration = end - start
    peak_mem_mb = peak_mem / (1024 * 1024)
    
    times_skmob.append(duration)
    peak_memories_skmob.append(peak_mem_mb)
    
    print(f"scikit-mobility Round {i+1}: {duration:.4f} seconds | Peak Overhead: {peak_mem_mb:.2f} MB")

print(f"scikit-mobility Average Time: {sum(times_skmob)/len(times_skmob):.4f} s")
print(f"scikit-mobility Minimum Time: {min(times_skmob):.4f} s")
print(f"scikit-mobility Average Peak Overhead: {sum(peak_memories_skmob)/len(peak_memories_skmob):.2f} MB")

Warming up scikit-mobility...


100%|█████████▉| 51378/51406 [01:16<00:00, 849.53it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|█████████▉| 51380/51406 [03:23<00:00, 268.90it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return geta

scikit-mobility Round 1: 212.9162 seconds | Peak Overhead: 725.20 MB


100%|█████████▉| 51400/51406 [03:15<00:00, 361.05it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|██████████| 51406/51406 [03:15<00:00, 262.30it/s]


scikit-mobility Round 2: 207.8109 seconds | Peak Overhead: 725.20 MB


...

scikit-mobility Round 3: 207.2124 seconds | Peak Overhead: 725.20 MB


100%|█████████▉| 51378/51406 [03:16<00:00, 338.57it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|██████████| 51406/51406 [03:16<00:00, 260.97it/s]


scikit-mobility Round 4: 206.5643 seconds | Peak Overhead: 725.20 MB


100%|█████████▉| 51385/51406 [03:19<00:00, 371.74it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|██████████| 51406/51406 [03:19<00:00, 257.69it/s]


scikit-mobility Round 5: 209.3129 seconds | Peak Overhead: 725.20 MB
scikit-mobility Average Time: 208.7633 s
scikit-mobility Minimum Time: 206.5643 s
scikit-mobility Average Peak Overhead: 725.20 MB


In [7]:
try:
    from skmob.measures.individual import jump_lengths as rog_skmob
    from skmob import TrajDataFrame
except Exception as exc: 
    raise SystemExit("Unable to import scikit-mobility. Install it via pip.")

# Prepare the data exactly as scikit-mobility expects
df_skmob = df.rename(columns={"latitude": "lat", "longitude": "lng", "check-in_time": "datetime"})

print("Warming up scikit-mobility...")
# Include TrajDataFrame creation in warmup
traj_df_warmup = TrajDataFrame(df_skmob, user_id="user", latitude="lat", longitude="lng")
_ = rog_skmob(traj_df_warmup)

times_skmob = []
peak_memories_skmob = []

for i in range(5):
    time.sleep(0.5)
    
    start = time.perf_counter()
    
    # Include required initialization in the timer, just like the real world
    traj_df = TrajDataFrame(df_skmob, user_id="user", latitude="lat", longitude="lng")
    _ = rog_skmob(traj_df)
    
    end = time.perf_counter()
    
    duration = end - start
    peak_mem_mb = peak_mem / (1024 * 1024)
    
    times_skmob.append(duration)
    
    print(f"scikit-mobility Round {i+1}: {duration:.4f} seconds")

print(f"scikit-mobility Average Time: {sum(times_skmob)/len(times_skmob):.4f} s")
print(f"scikit-mobility Minimum Time: {min(times_skmob):.4f} s")
# print(f"scikit-mobility Average Peak Overhead: {sum(peak_memories_skmob)/len(peak_memories_skmob):.2f} MB")

Warming up scikit-mobility...


100%|█████████▉| 51380/51406 [01:16<00:00, 862.51it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|█████████▉| 51333/51406 [01:16<00:00, 839.41it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return geta

scikit-mobility Round 1: 79.2687 seconds


100%|█████████▉| 51376/51406 [01:15<00:00, 817.62it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|██████████| 51406/51406 [01:15<00:00, 678.06it/s]


scikit-mobility Round 2: 78.8384 seconds


...

scikit-mobility Round 3: 85.1106 seconds


100%|█████████▉| 51394/51406 [01:15<00:00, 849.35it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|██████████| 51406/51406 [01:15<00:00, 676.71it/s]


scikit-mobility Round 4: 78.9877 seconds


100%|█████████▉| 51363/51406 [01:17<00:00, 853.70it/s]/home/gustavo/skmob2/.venv-skmob/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|██████████| 51406/51406 [01:17<00:00, 666.47it/s]


scikit-mobility Round 5: 80.2119 seconds
scikit-mobility Average Time: 80.4835 s
scikit-mobility Minimum Time: 78.8384 s


# Bar Plot Comparision

In [1]:
import json
import argparse
import numpy as np
import matplotlib.pyplot as plt
import os

def load_json(filepath):
    with open(filepath, 'r') as f:
        return json.load(f)

def generate_plots(file_original, file_optimized, target_sizes=['1M', '4M']):
    # Load JSON data
    data_orig = load_json(file_original)
    data_opt = load_json(file_optimized)
    
    # Create dictionaries mapping the size label (e.g., "1M") to its metrics
    dict_orig = {item['label']: item['metrics'] for item in data_orig['results']}
    dict_opt = {item['label']: item['metrics'] for item in data_opt['results']}
    
    for size in target_sizes:
        if size not in dict_orig or size not in dict_opt:
            print(f"Skipping {size}: Size label not found in both files.")
            continue
            
        metrics_orig = dict_orig[size]
        metrics_opt = dict_opt[size]
        
        # Find common metrics
        common_metrics = set(metrics_orig.keys()).intersection(set(metrics_opt.keys()))
        
        plot_data = []
        for m in common_metrics:
            t_orig = metrics_orig[m].get('average_seconds')
            t_opt = metrics_opt[m].get('average_seconds')
            
            # Skip if any run errored out (e.g., None values) or if optimized time is 0
            if t_orig is None or t_opt is None or t_opt == 0:
                continue
                
            speedup = t_orig / t_opt
            plot_data.append((m, t_orig, t_opt, speedup))
            
        if not plot_data:
            print(f"No valid overlapping data found for {size}.")
            continue
            
        # Sort by original time (ascending) so the longest tasks appear at the top of the graph
        plot_data.sort(key=lambda x: x[1])
        labels, orig_times, opt_times, speedups = zip(*plot_data)
        
        # Setup the plot
        y = np.arange(len(labels))
        height = 0.35  # Bar width
        
        fig, ax = plt.subplots(figsize=(12, max(6, len(labels) * 0.6)))
        
        # Plot horizontal bars
        rects1 = ax.barh(y - height/2, orig_times, height, label='Original', color='#ff9f43')
        rects2 = ax.barh(y + height/2, opt_times, height, label='Optimized', color='#00d2d3')
        
        # Format axes
        ax.set_xlabel('Average Execution Time (Seconds)', fontsize=11, fontweight='bold')
        ax.set_title(f'Performance Comparison: {size} Rows', fontsize=14, fontweight='bold', pad=15)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=10)
        ax.legend(loc='lower right', fontsize=11)
        
        # Add 15% padding to the right for text labels
        max_x = max(orig_times)
        ax.set_xlim(0, max_x * 1.15)
        
        # Add texts to bars
        for i in range(len(labels)):
            t_orig = orig_times[i]
            t_opt = opt_times[i]
            sp = speedups[i]
            
            # Text at the end of the bars (absolute times)
            # Offset slightly to the right of the bar tip
            offset = max_x * 0.01 
            ax.text(t_orig + offset, i - height/2, f'{t_orig:.3f}s', va='center', ha='left', fontsize=9, color='#d35400', fontweight='bold')
            ax.text(t_opt + offset, i + height/2, f'{t_opt:.3f}s', va='center', ha='left', fontsize=9, color='#01a3a4', fontweight='bold')
            
            # Speedup text in the middle
            # Placed visually halfway across the original (longer) bar, vertically centered between the two bars
            middle_x = t_orig / 2
            ax.text(middle_x, i, f'{sp:.1f}x Speedup', va='center', ha='center', 
                    fontsize=9, fontweight='bold', color='black',
                    bbox=dict(facecolor='white', alpha=0.85, edgecolor='#bdc3c7', boxstyle='round,pad=0.3'))
        
        # Clean up borders
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.xaxis.grid(True, linestyle='--', alpha=0.6)
        ax.set_axisbelow(True)
        
        plt.tight_layout()
        
        # Save and show
        output_filename = f'benchmark_comparison_{size}.png'
        plt.savefig(output_filename, dpi=300)
        print(f"Saved plot for {size} to {output_filename}")
        plt.close()

# if __name__ == "__main__":
#     parser = argparse.ArgumentParser(description="Generate bar plots from benchmark JSON files.")
#     parser.add_argument("original_json", help="Path to the original skmob benchmark JSON file")
#     parser.add_argument("optimized_json", help="Path to the optimized skmob2 benchmark JSON file")
    
#     args = parser.parse_args()
    
#     if not os.path.exists(args.original_json) or not os.path.exists(args.optimized_json):
#         print("Error: One or both JSON files were not found.")
#     else:
#         # You can add or remove sizes here (e.g. '100k') if you want more plots
#         generate_plots(args.original_json, args.optimized_json, target_sizes=['1M', '4M'])

In [7]:
import os

os.listdir("./results")

['skmob2_visits_memory_pandas.json',
 'skmob2_spatial_speed_pandas.json',
 'skmob2_spatial_memory_polars.json',
 'skmob_privacy_speed_prebuilt_tdf.json',
 'skmob_spatial_speed_prebuilt_tdf.json',
 'skmob2_privacy_speed_polars.json',
 'skmob2_privacy_speed_pandas.json',
 'skmob_models_speed.json',
 'skmob2_visits_memory_polars.json',
 'skmob2_spatial_speed_polars.json',
 'skmob2_visits_speed_pandas.json',
 'skmob2_visits_speed_polars.json',
 'skmob2_spatial_memory_pandas.json',
 'skmob2_spatial_speed.json',
 'logs',
 'skmob2_models_speed.json']

In [9]:
from pathlib import Path
# Resolve a repository-relative results directory robustly
base = Path.cwd()
# Prefer tests/benchmarks/results when present, fall back to ./results
if (base / 'tests' / 'benchmarks' / 'results').exists():
    results_dir = base / 'tests' / 'benchmarks' / 'results'
else:
    results_dir = base / 'results'
generate_plots(str(results_dir / 'skmob_spatial_speed_prebuilt_tdf.json'),
               str(results_dir / 'skmob2_spatial_speed_pandas.json'),
               target_sizes=['1M', '4M'])

Saved plot for 1M to benchmark_comparison_1M.png
Saved plot for 4M to benchmark_comparison_4M.png
